In [2]:
import pandas as pd 
import numpy as np 

In [ ]:
file_path =r"C:\Users\balaj\OneDrive\Desktop\TELANGANA PDS ANALYTICS\Data\processed\monthly_unified.csv"

monthly_data = pd.read_csv(file_path)

print("monthly data shape :", monthly_data.shape)
print("monthly data first five rows and columsn :", monthly_data.head())


#define the shop keys 

shop_key = ["distCode", "shopNo"]



In [4]:
#create monthly ratios

#utilization ratio = no of trans / total rcs 

#error can stop by replacing 0 to nan 

monthly_ratio = (monthly_data["noOfTrans"]/monthly_data["totalRcs"].replace(0,np.nan)) # stored temporarily 
portability_ratio = ( monthly_data["otherShopTransCnt"] / monthly_data["noOfTrans"].replace(  0, np.nan
    )
)


rice_columns = [
    "riceAfsc",
    "riceFsc",
    "riceAap"
]

monthly_data["total_rice"]=monthly_data[rice_columns].fillna(0).sum(axis=1)#cumulative aggregation 



In [ ]:
# create one feature row per shop 

shop_features  =(
    monthly_data.groupby(shop_key, as_index=False  # groupby aggregation pattern , splits dataset based on shop key
                         
)
.agg( 
    reporting_months=(
        "noOfTrans",
        "count"                                  # all following these are feature engineering
    ),
    total_transactions = (
        "noOfTrans",
        "sum"
    ),
    average_transactions=(
        "noOfTrans",
        "mean"
    ),
    transaction_volatility =(
        "noOfTrans",
        "std"
    ),
    average_cards=(
            "totalRcs",
            "mean"
        ),

    average_utilization=(
            "utilization_ratio",
            "mean"
        ),

    average_portability_ratio=(
            "portability_ratio",
            "mean"
        ),

    total_portability_transactions=(
            "otherShopTransCnt",
            "sum"
        ),

    average_monthly_rice=(
            "total_rice",
            "mean"
        ),

    average_monthly_wheat=(
            "wheat",
            "mean"
        ),

    average_rice_wheat_ratio=(
            "rice_wheat_ratio",
            "mean"
)
)
)

print(shop_features.shape)
print(shop_features.head())


In [ ]:
#shop_features.isna() returns a series like missing counts , columns names 
#converting into dataframe makes earier to add, sort , print a tabulr format 



missing_report = pd.DataFrame({
    "missing_count": (
        shop_features.isna().sum()
    ),

    "missing_percentage": (
        shop_features.isna().mean()
        * 100
    ).round(2)
})

#boolean indexing where explicity reassigning into filetered version 

missing_report = missing_report[
    missing_report["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

print(missing_report)


model_data = shop_features[
    shop_features["reporting_months"] >= 6
].copy()

print(
    "All shops:",
    len(shop_features)
)

print(
    "Shops retained for clustering:",
    len(model_data)
)

print(
    "Shops excluded:",
    len(shop_features) - len(model_data)
)

MODEL_COLUMNS = [
    "average_transactions",
    "transaction_volatility",
    "average_cards",
    "average_utilization",
    "average_portability_ratio",
    "average_monthly_rice",
    "average_monthly_wheat",
    "average_rice_wheat_ratio"
]

model_data[
    MODEL_COLUMNS
].isna().sum()




In [ ]:
X = model_data[
    MODEL_COLUMNS
].copy()

X_filled = X.fillna(
    X.median()
)

print(
    "Missing values after filling:",
    X_filled.isna().sum().sum()
)

In [ ]:
print(X.isna().sum())